# Phase 1: Simulation Engine
This notebook contains the pure order book mechanics.

In [ ]:
import heapq
import random
import pandas as pd
import numpy as np
import pandas_ta as ta
import plotly.graph_objects as go
from typing import List, Optional

print("Libraries loaded successfully!")

### 1. Order Book & Trader

In [ ]:
class OrderBook:
    def __init__(self):
        self.bids = []  # Max-heap: (-price, size, order_id)
        self.asks = []  # Min-heap: (price, size, order_id)
        self.order_id_counter = 0
        self.last_price = 100.0
        
    def add_maker_order(self, side, price, size):
        self.order_id_counter += 1
        if side == 'buy':
            heapq.heappush(self.bids, (-price, size, self.order_id_counter))
        else:
            heapq.heappush(self.asks, (price, size, self.order_id_counter))
            
    def execute_taker_order(self, side, size):
        executed_value = 0.0
        executed_size = 0
        
        if side == 'buy':
            while size > 0 and self.asks:
                best_ask_price, ask_size, order_id = self.asks[0]
                if size >= ask_size:
                    heapq.heappop(self.asks)
                    executed_size += ask_size
                    executed_value += best_ask_price * ask_size
                    size -= ask_size
                    self.last_price = best_ask_price
                else:
                    self.asks[0] = (best_ask_price, ask_size - size, order_id)
                    executed_size += size
                    executed_value += best_ask_price * size
                    self.last_price = best_ask_price
                    size = 0
        else:
            while size > 0 and self.bids:
                neg_best_bid_price, bid_size, order_id = self.bids[0]
                best_bid_price = -neg_best_bid_price
                if size >= bid_size:
                    heapq.heappop(self.bids)
                    executed_size += bid_size
                    executed_value += best_bid_price * bid_size
                    size -= bid_size
                    self.last_price = best_bid_price
                else:
                    self.bids[0] = (neg_best_bid_price, bid_size - size, order_id)
                    executed_size += size
                    executed_value += best_bid_price * size
                    self.last_price = best_bid_price
                    size = 0
                    
        return (executed_value / executed_size) if executed_size > 0 else self.last_price

class RandomTrader:
    def __init__(self, maker_ratio=0.5, max_deviation=0.01):
        self.maker_ratio = maker_ratio
        self.max_deviation = max_deviation
        
    def act(self, order_book):
        is_maker = random.random() < self.maker_ratio
        side = 'buy' if random.random() < 0.5 else 'sell'
        size = random.randint(1, 10)
        
        if is_maker:
            ref_price = order_book.last_price
            deviation = ref_price * self.max_deviation
            if side == 'buy':
                price = ref_price - random.uniform(0, deviation)
            else:
                price = ref_price + random.uniform(0, deviation)
            order_book.add_maker_order(side, round(price, 2), size)
            return None
        else:
            return order_book.execute_taker_order(side, size)

### 2. Tick Engine
Runs the simulation for N ticks and aggregates the price history into OHLCV candles.

In [ ]:
def run_simulation(ticks=10000, traders_per_tick=10, maker_ratio=0.7):
    order_book = OrderBook()
    traders = [RandomTrader(maker_ratio=maker_ratio, max_deviation=0.02) for _ in range(50)]
    
    # Pre-seed the order book with some liquidity
    for _ in range(100):
        random.choice(traders).act(order_book)
        
    price_history = []
    
    for tick in range(ticks):
        tick_prices = []
        for _ in range(traders_per_tick):
            trader = random.choice(traders)
            exec_price = trader.act(order_book)
            if exec_price is not None:
                tick_prices.append(exec_price)
                
        if not tick_prices:
            tick_prices = [order_book.last_price]
            
        price_history.append({
            'tick': tick,
            'open': tick_prices[0],
            'high': max(tick_prices),
            'low': min(tick_prices),
            'close': tick_prices[-1],
            'volume': len(tick_prices) # Proxy for volume
        })
        
    df = pd.DataFrame(price_history)
    return df, order_book

### 3. Run Simulation & Phase 2: Indicators

In [ ]:
# Run simulation
df, ob = run_simulation(ticks=1000, traders_per_tick=20, maker_ratio=0.8)
print(f"Simulation finished. DataFrame shape: {df.shape}")

# Calculate Indicators using pandas-ta
df.ta.sma(length=20, append=True)
df.ta.sma(length=50, append=True)
df.ta.macd(append=True)
df.ta.rsi(length=14, append=True)
df.ta.bbands(append=True)

df.tail()

### 4. Visualization
Plot the emergent chart using Plotly.

In [ ]:
fig = go.Figure(data=[go.Candlestick(x=df['tick'],
                open=df['open'],
                high=df['high'],
                low=df['low'],
                close=df['close'],
                name='Price')])

# Add SMA 20
fig.add_trace(go.Scatter(x=df['tick'], y=df['SMA_20'], line=dict(color='orange', width=1.5), name='SMA 20'))
# Add SMA 50
fig.add_trace(go.Scatter(x=df['tick'], y=df['SMA_50'], line=dict(color='blue', width=1.5), name='SMA 50'))

fig.update_layout(title='Simulated Market Prices with Emergent Patterns',
                  xaxis_title='Tick',
                  yaxis_title='Price',
                  xaxis_rangeslider_visible=False)
fig.show()